# 02 — Canonical time-series EDA

This notebook studies calibration telemetry only. Its purpose is to make a
small number of documented modelling decisions—not to generate a general
profiling report.

It never reads `SPEC-EVAL` or holdout telemetry.

## 1. Setup and sector switch

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or (
        "/content/drive/MyDrive/anomaly_detection"
        if IN_COLAB else Path.home() / "anomaly_detection_data"
    )
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run1",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

import duckdb
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import STL

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

EDA_VERSION = "1.3.0"
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{SECTOR}_eda_v1_3_run1")
RUN_ROOT = (
    DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}"
    / SECTOR / CANONICAL_RUN_ID
)
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
SPLIT_ROOT = RUN_ROOT / "SPLITS"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / EDA_RUN_ID

# These are transparent research durations, not hidden model branches.
TIME_SETTINGS = {
    "telecom": {
        "rolling_windows_seconds": {
            "short": 24 * 60 * 60,
            "long": 7 * 24 * 60 * 60,
        },
        "minimum_history_seconds": 6 * 60 * 60,
        "persistence_seconds": 30 * 60,
        "recovery_seconds": 60 * 60,
    },
    "petrobras_3w": {
        "rolling_windows_seconds": {
            "short": 2 * 60,
            "long": 30 * 60,
        },
        "minimum_history_seconds": 30,
        "persistence_seconds": 15,
        "recovery_seconds": 30,
    },
}
MAX_PLOT_METRICS = int(os.getenv("EDA_MAX_PLOT_METRICS", "6"))
MAX_SERIES_POINTS = int(os.getenv("EDA_MAX_SERIES_POINTS", "5000"))

display(pd.Series({
    "sector": SECTOR,
    "canonical_input": str(CORE_ROOT),
    "eda_output": str(EDA_ROOT),
    "analysis_partition": "calibration",
}, name="value").to_frame())

## 2. Load calibration telemetry

In [ ]:
if not CORE_ROOT.is_dir():
    raise FileNotFoundError(f"Run Notebook 01B first: {CORE_ROOT}")

manifest = read_json(CORE_ROOT / "manifest.json")
if manifest["sector"] != SECTOR:
    raise ValueError("Canonical sector does not match the requested sector")
canonical_build_ts = pd.Timestamp(
    (CORE_ROOT / "manifest.json").stat().st_mtime, unit="s", tz="UTC"
)
display(pd.Series({
    "canonical_run_id": CANONICAL_RUN_ID,
    "canonical_build_ts": canonical_build_ts,
}, name="value").to_frame())
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
connection = duckdb.connect()
connection.execute("SET memory_limit = ?", [os.getenv("ANOMALY_DUCKDB_MEMORY_LIMIT", "3GB")])
connection.execute("SET threads = ?", [int(os.getenv("ANOMALY_DUCKDB_THREADS", "2"))])
telemetry_glob = str(CORE_ROOT / "telemetry" / "*.parquet").replace("'", "''")
connection.execute(
    f"CREATE VIEW telemetry_all AS SELECT * FROM read_parquet('{telemetry_glob}')"
)

time_path = SPLIT_ROOT / "time_partitions.parquet"
entity_path = SPLIT_ROOT / "entity_partitions.parquet"
if time_path.is_file():
    partitions = pd.read_parquet(time_path)
    calibration = partitions.loc[partitions["partition"].eq("calibration")]
    if len(calibration) != 1:
        raise ValueError("Expected one calibration time partition")
    start = pd.to_datetime(calibration.iloc[0]["start_ts"], utc=True)
    end = pd.to_datetime(calibration.iloc[0]["end_ts"], utc=True)
    connection.execute(f"""
        CREATE VIEW telemetry AS
        SELECT * FROM telemetry_all
        WHERE event_ts >= TIMESTAMPTZ '{start.isoformat()}'
          AND event_ts < TIMESTAMPTZ '{end.isoformat()}'
    """)
    PRIMARY_SPLIT = "time"
else:
    partitions = pd.read_parquet(entity_path)
    calibration_entities = partitions.loc[
        partitions["partition"].eq("calibration"), ["entity_id"]
    ].astype(str)
    connection.register("calibration_entities", calibration_entities)
    connection.execute("""
        CREATE VIEW telemetry AS
        SELECT t.* FROM telemetry_all AS t
        JOIN calibration_entities AS c
          ON CAST(t.entity_id AS VARCHAR) = c.entity_id
    """)
    PRIMARY_SPLIT = "entity"

def sql(query):
    return connection.execute(query).df()

display(catalogue)

## 3. Structure, quality and distributions

In [ ]:
inventory = sql("""
    SELECT count(*) AS rows,
           count(DISTINCT entity_id) AS entities,
           count(DISTINCT episode_id) AS episodes,
           count(DISTINCT metric_id) AS metrics,
           min(event_ts) AS first_ts,
           max(event_ts) AS last_ts
    FROM telemetry
""")

metric_summary = sql("""
    SELECT metric_id,
           count(*) AS rows,
           count(DISTINCT entity_id) AS entities,
           avg(CASE WHEN quality_code = 'invalid' OR value IS NULL
                    OR NOT isfinite(value) THEN 1.0 ELSE 0.0 END) AS invalid_rate,
           avg(CASE WHEN quality_code = 'clipped' THEN 1.0 ELSE 0.0 END) AS clipped_rate,
           approx_quantile(value, 0.01) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS q01,
           approx_quantile(value, 0.50) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS median,
           approx_quantile(value, 0.99) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS q99,
           max(abs(value)) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS max_abs,
           skewness(value) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS skewness
    FROM telemetry
    GROUP BY metric_id
""")

series_quality = sql("""
    WITH per_series AS (
        SELECT metric_id, entity_id, episode_id, count(*) AS rows,
               avg(CASE WHEN quality_code <> 'invalid' AND value IS NOT NULL
                        AND isfinite(value) THEN 1.0 ELSE 0.0 END) AS valid_rate
        FROM telemetry
        GROUP BY metric_id, entity_id, episode_id
    )
    SELECT metric_id, count(*) AS series_count,
           approx_quantile(valid_rate, 0.10) AS valid_rate_p10,
           approx_quantile(valid_rate, 0.50) AS valid_rate_p50,
           approx_quantile(valid_rate, 0.90) AS valid_rate_p90
    FROM per_series
    GROUP BY metric_id
""")
metric_summary = (
    metric_summary.merge(series_quality, on="metric_id", how="left")
    .merge(catalogue, on="metric_id", how="left")
)

duplicate_keys = sql("""
    SELECT count(*) - count(DISTINCT (entity_id, episode_id, metric_id, event_ts)) AS duplicates
    FROM telemetry
""").iloc[0, 0]

display(inventory.T.rename(columns={0: "value"}))
display(metric_summary[[
    "metric_id", "measurement_kind", "unit", "sampling_mode",
    "expected_cadence_seconds", "rows", "entities", "invalid_rate",
    "clipped_rate", "valid_rate_p10", "valid_rate_p50",
    "q01", "median", "q99", "max_abs", "skewness",
]].round(4))
print("Duplicate canonical keys:", int(duplicate_keys))

figure, axes = plt.subplots(1, 2, figsize=(12, 3.5))
metric_summary.set_index("metric_id")[["invalid_rate", "clipped_rate"]].plot.bar(ax=axes[0])
axes[0].set_title("Calibration quality rates")
axes[0].set_ylabel("fraction of rows")
axes[1].bar(metric_summary["metric_id"], metric_summary["skewness"].clip(-10, 10))
axes[1].tick_params(axis="x", rotation=90)
axes[1].set_title("Skewness (clipped to ±10 for display)")
figure.tight_layout()
plt.show()

## 4. Representative distributions and time series

In [ ]:
plot_metrics = (
    metric_summary.sort_values("rows", ascending=False)
    .groupby("measurement_kind", sort=False).head(1)["metric_id"].tolist()
)
plot_metrics += [
    metric for metric in metric_summary.sort_values("rows", ascending=False)["metric_id"]
    if metric not in plot_metrics
]
plot_metrics = plot_metrics[:MAX_PLOT_METRICS]

sample = sql("""
    SELECT entity_id, episode_id, metric_id, event_ts, value
    FROM (
        SELECT entity_id, episode_id, metric_id, event_ts, value
        FROM telemetry
        WHERE quality_code <> 'invalid' AND value IS NOT NULL AND isfinite(value)
    ) USING SAMPLE reservoir(120000 ROWS) REPEATABLE (42)
""")
sample["event_ts"] = pd.to_datetime(sample["event_ts"], utc=True)

representative = sql(f"""
    WITH series_sizes AS (
        SELECT metric_id, entity_id, episode_id, count(*) AS rows
        FROM telemetry
        WHERE quality_code <> 'invalid' AND value IS NOT NULL
        GROUP BY metric_id, entity_id, episode_id
    ), chosen AS (
        SELECT metric_id, entity_id, episode_id FROM series_sizes
        QUALIFY row_number() OVER (
            PARTITION BY metric_id
            ORDER BY rows DESC, entity_id, episode_id
        ) = 1
    )
    SELECT t.entity_id, t.episode_id, t.metric_id, t.event_ts, t.value
    FROM telemetry AS t
    JOIN chosen AS c USING (metric_id, entity_id, episode_id)
    WHERE t.metric_id IN ({','.join(repr(value) for value in plot_metrics)})
      AND t.quality_code <> 'invalid' AND t.value IS NOT NULL
    QUALIFY row_number() OVER (PARTITION BY t.metric_id ORDER BY t.event_ts)
            <= {MAX_SERIES_POINTS}
    ORDER BY t.metric_id, t.event_ts
""")
representative["event_ts"] = pd.to_datetime(representative["event_ts"], utc=True)

kind = catalogue.set_index("metric_id")["measurement_kind"].to_dict()
for metric_id in plot_metrics:
    values = pd.to_numeric(
        sample.loc[sample["metric_id"].eq(metric_id), "value"], errors="coerce"
    ).dropna()
    series = representative.loc[
        representative["metric_id"].eq(metric_id), ["event_ts", "value"]
    ].drop_duplicates("event_ts").sort_values("event_ts").set_index("event_ts")["value"]
    if values.empty or series.empty:
        continue
    if kind[metric_id] == "interval_count":
        values = np.log1p(values.clip(lower=0))
        distribution_label = "log1p(value)"
    elif kind[metric_id] == "cumulative_counter":
        values = series.diff().dropna()
        distribution_label = "representative-episode increment"
    else:
        distribution_label = "value"

    figure, axes = plt.subplots(1, 2, figsize=(12, 3.2))
    axes[0].hist(values, bins=40, color="#4472C4")
    axes[0].set_title(f"{metric_id}: {distribution_label}")
    axes[1].plot(series.index, series, linewidth=0.8, label="value")
    if kind[metric_id] != "discrete_state" and len(series) >= 20:
        window = max(5, min(96, len(series) // 20))
        axes[1].plot(series.rolling(window).median(), color="black", label="rolling median")
        axes[1].legend()
    axes[1].set_title(f"{metric_id}: representative series")
    figure.tight_layout()
    plt.show()

## 5. Autocorrelation, seasonality and cross-metric dependence

In [ ]:
cadence = catalogue.set_index("metric_id")["expected_cadence_seconds"].to_dict()
sampling = catalogue.set_index("metric_id")["sampling_mode"].to_dict()
seasonality_rows = []

def longest_regular_segment(series, cadence_seconds):
    regular = series.asfreq(pd.Timedelta(seconds=float(cadence_seconds)))
    regular = regular.interpolate(limit=2, limit_area="inside")
    valid = regular.notna()
    if not valid.any():
        return regular.iloc[:0]
    runs = valid.ne(valid.shift()).cumsum()
    longest = runs.loc[valid].value_counts().idxmax()
    return regular.loc[runs.eq(longest)].dropna()

for metric_id in plot_metrics:
    series = representative.loc[
        representative["metric_id"].eq(metric_id), ["event_ts", "value"]
    ].drop_duplicates("event_ts").sort_values("event_ts").set_index("event_ts")["value"]
    series = pd.to_numeric(series, errors="coerce").dropna()
    metric_cadence = cadence.get(metric_id)
    if pd.isna(metric_cadence) or kind[metric_id] == "discrete_state":
        continue
    if kind[metric_id] == "interval_count":
        series = np.log1p(series.clip(lower=0))
    elif kind[metric_id] == "cumulative_counter":
        series = series.diff().dropna()

    regular = longest_regular_segment(series, metric_cadence)
    if len(regular) < 60 or regular.nunique() < 3:
        continue

    figure, axis = plt.subplots(figsize=(8, 3))
    plot_acf(regular.iloc[:5000], lags=min(60, len(regular) // 3), zero=False, ax=axis)
    axis.set_title(f"{metric_id}: ACF")
    figure.tight_layout()
    plt.show()

    status, strength = "not_applicable", np.nan
    if sampling.get(metric_id) == "periodic":
        period = round(86400 / float(metric_cadence))
        if period >= 2 and len(regular) >= 6 * period:
            result = STL(regular, period=period, robust=True).fit()
            denominator = np.nanvar(result.seasonal + result.resid)
            strength = max(0, 1 - np.nanvar(result.resid) / denominator) if denominator else 0
            status = "evaluated"
        else:
            status = "insufficient_cycles"
    seasonality_rows.append({
        "metric_id": metric_id,
        "lag1_acf": regular.autocorr(1),
        "daily_seasonal_strength": strength,
        "seasonality_status": status,
    })

seasonality = pd.DataFrame(seasonality_rows)
display(seasonality.round(4))

continuous = catalogue.loc[
    ~catalogue["measurement_kind"].eq("discrete_state"), "metric_id"
].astype(str).tolist()
correlation_sample = sql("""
    WITH coverage AS (
        SELECT entity_id, episode_id, count(DISTINCT metric_id) AS metrics,
               count(*) AS rows
        FROM telemetry
        WHERE quality_code <> 'invalid' AND value IS NOT NULL
        GROUP BY entity_id, episode_id
    ), chosen AS (
        SELECT entity_id, episode_id FROM coverage
        ORDER BY metrics DESC, rows DESC, entity_id, episode_id LIMIT 1
    )
    SELECT t.event_ts, t.metric_id, t.value
    FROM telemetry AS t JOIN chosen AS c USING (entity_id, episode_id)
    WHERE t.quality_code <> 'invalid' AND t.value IS NOT NULL
    QUALIFY row_number() OVER (PARTITION BY t.metric_id ORDER BY t.event_ts) <= 5000
""")
pivot = correlation_sample.loc[
    correlation_sample["metric_id"].isin(continuous)
].pivot_table(
    index="event_ts",
    columns="metric_id", values="value", aggfunc="first",
)
level_correlation = pivot.corr(method="spearman")
difference_correlation = pivot.diff().corr(method="spearman")

figure, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, matrix, title in zip(
    axes, [level_correlation, difference_correlation],
    ["Representative episode: level correlation",
     "Representative episode: change correlation"],
):
    image = axis.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm")
    axis.set_xticks(range(len(matrix)), matrix.columns, rotation=90)
    axis.set_yticks(range(len(matrix)), matrix.index)
    axis.set_title(title)
figure.colorbar(image, ax=axes, shrink=0.75)
plt.show()

## 6. Freeze the small modelling decision record

In [ ]:
cadence_values = [int(value) for value in sorted(pd.to_numeric(
    catalogue["expected_cadence_seconds"], errors="coerce"
).dropna().astype(int).unique())]
if not cadence_values:
    raise ValueError("The baseline requires a declared metric cadence")
base_cadence = int(np.median(cadence_values))
settings = TIME_SETTINGS[SECTOR]
rolling_windows = settings["rolling_windows_seconds"]
long_window_seconds = max(rolling_windows.values())
persistence_observations = max(
    1, round(settings["persistence_seconds"] / base_cadence)
)
decisions = {
    "eda_version": EDA_VERSION,
    "sector": SECTOR,
    "analysis_partition": "calibration",
    "canonical_fingerprint": manifest["fingerprint"],
    "primary_split": PRIMARY_SPLIT,
    "base_cadence_seconds": base_cadence,
    "cadence_values_seconds": cadence_values,
    "rolling_windows_seconds": rolling_windows,
    "rolling_window_observations": max(4, round(long_window_seconds / base_cadence)),
    "minimum_history_seconds": settings["minimum_history_seconds"],
    "minimum_history_observations": max(3, round(settings["minimum_history_seconds"] / base_cadence)),
    "persistence_seconds": settings["persistence_seconds"],
    "persistence_observations": persistence_observations,
    "recovery_observations": max(1, round(settings["recovery_seconds"] / base_cadence)),
    "measurement_kind_transformations": {
        "gauge": "long_scaled_change_and_short_long_robust_z",
        "bounded_fraction": "level_long_scaled_change_and_short_long_robust_z",
        "interval_count": "log1p_then_long_scaled_change_and_short_long_robust_z",
        "cumulative_counter": "nonnegative_increment_and_reset",
        "discrete_state": "level_and_state_change",
    },
    "clipped_value_decision": "retain as observed saturated values; report clipped rate separately",
    "seasonal_features_enabled": False,
    "seasonal_feature_reason": "EDA evidence is reported; the first baseline remains intentionally small",
}

source_metadata = manifest.get("source_metadata", {})
longest_fault = source_metadata.get(
    "longest_declared_fault_duration_seconds"
)
display(pd.Series({
    "long_window_seconds": long_window_seconds,
    "longest_declared_fault_duration_seconds": (
        longest_fault if longest_fault is not None
        else "not available in label-free source metadata"
    ),
}, name="duration").to_frame())

with new_output_directory(EDA_ROOT) as output:
    metric_summary.to_csv(output / "eda_summary.csv", index=False)
    write_json(output / "eda_decisions.json", decisions)

display(pd.Series(decisions, name="decision").to_frame())
print("Saved:", EDA_ROOT)
print("Next: 03_EVALUATION_CONTRACT.ipynb")
connection.close()